# Complete Thesis Pipeline: From Raw Data to Conclusions

**TCC Title**: Data-Driven Public Compliance — Correlation between Federal Transfers, Corruption Proxies and Municipal Socioeconomic Indicators in Brazil

**Author**: Enok Antônio de Jesus
**Advisor**: Prof. Dr. Carlos Nabil Ghobril
**Institution**: USP / ESALQ — MBA in Data Science & Analytics
**Date**: 2026

---

## Research Question

> **Does corruption (or poor use of public money) contribute to lower HDI indicators in Brazilian municipalities, impacting the socioeconomic condition of society as a whole?**

---

## Abstract

This notebook executes the full empirical pipeline of the thesis, from raw public data ingestion to final conclusions. It follows a **Medallion Architecture** (Databricks, 2023; Armbrust et al., 2020) — Bronze (raw), Silver (normalized), Gold (analysis-ready) — and applies a progression of analytical methods in accordance with accepted practice in applied econometrics and data science (James et al., 2021; Hastie et al., 2009; Kuhn & Johnson, 2013).

The empirical strategy combines **descriptive EDA**, **OLS regression with regional controls** (Fox, 2015; Montgomery et al., 2012), **regularized predictive modelling** (Hastie et al., 2009), **K-means clustering with PCA visualization** (Hair et al., 2018; James et al., 2021), and a **novel cluster-stratified correlation analysis** between a corruption proxy (sanctions per BRL transferred) and HDI components — a stratification motivated by the heterogeneity of Brazilian municipalities (Ferraz & Finan, 2008; Rose-Ackerman & Palifka, 2016).

---

## Notebook Structure

1. **Setup & reproducibility** — environment, seeds, paths
2. **Bronze Layer** — raw data sources and ingestion logic
3. **Silver Layer** — normalization, deflation, star schema
4. **Gold Layer** — analysis-ready aggregations
5. **Exploratory Data Analysis (EDA)** — univariate and bivariate exploration
6. **Statistical Analysis** — correlations, OLS regression with regional dummies
7. **Machine Learning** — ElasticNet, Random Forest, feature importance
8. **Clustering Analysis** — K-means (K=4) with PCA visualization
9. **Corruption vs HDI Analysis** — cluster-stratified correlations, vulnerability index
10. **QGIS Map Generation** — GeoJSON export for geospatial visualization
11. **Dashboard Export** — Power BI / Tableau CSV outputs
12. **Conclusions** — findings, limitations, policy implications
13. **References** — full bibliography in ABNT-compatible format


> **Thesis grain note (auto-inserted by migration):** the primary
> analytical grain for the thesis is now the **municipality**
> (`analysis_compliance_municipality`, N ~= 5,570). The state-level
> blocks below (`STATE-LEVEL DESCRIPTIVES`, `STATE-LEVEL CORRELATIONS`)
> are kept as presentation rollups (N=27) but are not the basis of
> the thesis's statistical claims anymore. See notebooks 01-03 for
> muni-level descriptives, OLS and ML. See notebook 04 for
> muni-level clustering and notebook 05 for the corruption x HDI
> analysis that already operates on munis.

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


---

## 1. Setup & Reproducibility

### 1.1 Methodological Justification

Reproducibility is a cornerstone of credible empirical research (Gandrud, 2020; Wickham & Grolemund, 2017). We therefore fix random seeds, declare dependencies explicitly, and persist all intermediate artefacts in the `data/` directory. The notebook is self-contained: all data is loaded from local Parquet files materialized by the ETL pipeline.

**Key references**:
- **Gandrud, C. (2020).** *Reproducible Research with R and RStudio* (3rd ed.). CRC Press.
- **Wickham, H., & Grolemund, G. (2017).** *R for Data Science*. O'Reilly.
- **McKinney, W. (2022).** *Python for Data Analysis* (3rd ed.). O'Reilly.


In [ ]:
import os, sys, json, warnings, tempfile, zipfile, shutil
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr, spearmanr
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)

project_root = Path('..').resolve()
bronze_dir = project_root / 'data' / 'bronze'
silver_dir = project_root / 'data' / 'silver'
gold_dir   = project_root / 'data' / 'gold'
output_dir = project_root / 'docs' / 'thesis_presentation_assets'
qgis_dir   = output_dir / 'qgis'
for d in [output_dir, qgis_dir]:
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.bbox'] = 'tight'

print(f"Project root: {project_root}")
print(f"Random seed: {SEED}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}  numpy: {np.__version__}")


In [ ]:
# ===== Sync data from S3 to local (if needed) =====
# Uses AWS_PROFILE from .env to mirror Bronze/Silver/Gold to the local data/ directory.
# Idempotent: only downloads files missing locally or with different sizes.
# Safe fallback: if AWS is unavailable, the notebook continues with existing local data.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env")
except ImportError:
    pass

try:
    from src.ingestion.s3_local_sync import sync_s3_prefix_to_local
    s3_bucket = os.getenv("S3_BUCKET_NAME", "enok-mba-thesis-datalake")
    for layer, local in [("bronze", bronze_dir), ("silver", silver_dir), ("gold", gold_dir)]:
        print(f"Syncing s3://{s3_bucket}/{layer}/ -> {local}")
        r = sync_s3_prefix_to_local(bucket=s3_bucket, prefix=f"{layer}/", local_dir=local)
        print(f"  [SYNC] downloaded={r['downloaded']} skipped={r['skipped']} errors={r['errors']}")
except Exception as exc:
    print(f"[WARN] S3 sync skipped: {exc}")
    print("       Continuing with existing local data. Run 00_etl_pipeline first to populate.")


---

## 2. Bronze Layer — Raw Data Ingestion

### 2.1 Methodological Justification — Medallion Architecture

The **Medallion Architecture** (Databricks, 2023) organizes data into three progressively refined layers:
- **Bronze**: Raw, immutable ingestion of source data (API responses, CSVs) preserved as-is for auditability.
- **Silver**: Cleaned, deduplicated, typed data in canonical schemas.
- **Gold**: Analysis-ready aggregations joined across sources.

This layered approach is recommended for data lakes (Armbrust et al., 2020) and aligns with classical data warehouse staging patterns (Inmon, 2005; Kimball & Ross, 2013). Bronze preservation is essential for **reproducibility** and **auditability** — regulators (TCU, 2023) and the Lei de Acesso à Informação (Brasil, 2011) emphasize data lineage for public-sector evidence.

### 2.2 Data Sources

| Source | Dataset | Period | Reference |
|--------|---------|--------|-----------|
| **IBGE SIDRA** | Census 2010 (pop, literacy, income, sanitation) | 2010 | IBGE (2010); IBGE (2024) |
| **IBGE SIDRA** | Census 2022 (pop, literacy, income, sanitation) | 2022 | IBGE (2022); IBGE (2024) |
| **Portal da Transparência** | Federal Transfers (Fundo-a-Fundo) | 2013–2022 | CGU (2024) |
| **Portal da Transparência** | Sanctions (CEIS, CNEP, CEPIM) | Cumulative | CGU (2024) |
| **BCB SGS** | IPCA monthly inflation index | 1980–2026 | STN (2024) |

**Legal basis**: Lei nº 12.527/2011 (LAI) (Brasil, 2011). No personal data ingested — only aggregated municipal statistics and public sanction records, consistent with LGPD (Brasil, 2018).


In [ ]:
bronze_sources = {
    'IBGE Census 2010': '5,565 municipalities x 4 domains',
    'IBGE Census 2022': '5,570 municipalities x 4 domains',
    'Federal Transfers': '~109 monthly files, 2013-12 to 2022-12',
    'CEIS Sanctions': '22,545 records (inidoneas)',
    'CNEP Sanctions': '1,625 records (entes privados)',
    'CEPIM Sanctions': '3,579 records (impedidas)',
    'IPCA Deflator': '47 yearly values (1980-2026)',
}

print("BRONZE LAYER - Data Sources Summary")
print("=" * 60)
for src, desc in bronze_sources.items():
    print(f"  - {src:30s} {desc}")


---

## 3. Silver Layer — Normalization & Deflation

### 3.1 Methodological Justification — Dimensional Modeling

The Silver layer materializes a **Star Schema** (Kimball & Ross, 2013) with dimension tables (`dim_municipalities`, `dim_inflation_index`) and fact tables (`fact_population`, `fact_income`, `fact_sanctions`, `fact_federal_transfers`, etc.). This pattern decouples reference data from events, enables efficient joins, and makes lineage explicit.

### 3.2 Deflation (Nominal → Real BRL)

Nominal monetary values are deflated to **2022 BRL** using the **IPCA** index (STN, 2024) — standard practice in Brazilian applied economics (Afonso, Schuknecht, & Tanzi, 2005):

$$ \text{real\_value}_t = \text{nominal\_value}_t \times \frac{\text{IPCA}_{2022}}{\text{IPCA}_t} $$

### 3.3 Municipality Code Standardization

All municipality codes are normalized to the **7-digit IBGE code** (IBGE, 2024).


In [ ]:
silver_tables = {}
silver_sources = [
    ('dim_municipalities',     'Municipality lookup'),
    ('dim_inflation_index',    'IPCA deflators'),
    ('fact_population',        'Population by year'),
    ('fact_literacy',          'Literacy rates'),
    ('fact_income',            'Income (nominal + real)'),
    ('fact_sanitation',        'Household counts'),
    ('fact_sanctions',         'Sanctions (CEIS+CNEP+CEPIM)'),
    ('fact_federal_transfers', 'Federal transfers (monthly)'),
]
print("SILVER LAYER - Loading normalized tables")
print("=" * 60)
for name, desc in silver_sources:
    p = silver_dir / name / 'data.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        silver_tables[name] = df
        print(f"  [OK] {name:28s} {len(df):>10,} rows - {desc}")
    else:
        print(f"  [!!] {name:28s} NOT FOUND")


In [ ]:
if 'dim_municipalities' in silver_tables:
    df_muni = silver_tables['dim_municipalities']
    print(f"Municipalities: {len(df_muni):,}")
    if 'state_code' in df_muni.columns:
        print(f"  States: {df_muni['state_code'].nunique()}")
    if 'region' in df_muni.columns:
        print(f"  Regions: {df_muni['region'].nunique()}")

if 'fact_population' in silver_tables:
    df_pop = silver_tables['fact_population']
    if 'year' in df_pop.columns and 'total_population' in df_pop.columns:
        print(f"\nPopulation by year (Brazil, millions):")
        for yr, p in (df_pop.groupby('year')['total_population'].sum() / 1e6).items():
            print(f"  {int(yr)}: {p:6.2f}M")

if 'fact_sanctions' in silver_tables:
    df_san = silver_tables['fact_sanctions']
    print(f"\nSanctions total: {len(df_san):,}")
    if 'registry_type' in df_san.columns:
        for reg, cnt in df_san['registry_type'].value_counts().items():
            print(f"  {reg}: {cnt:,}")


---

## 4. Gold Layer — Analysis-Ready Aggregations

### 4.1 Methodological Justification

The Gold layer materializes **denormalized, analysis-oriented datasets** that flatten dimensions into fact rows to minimize join overhead for statistical and ML workloads (Kleppmann, 2017).

| Dataset | Purpose | Grain |
|---------|---------|-------|
| `agg_municipality_socioeconomic` | Municipal feature vector (2010/2022 + deltas) | 1 row / municipality |
| `agg_state_summary` | State-level aggregates | 1 row / state |
| `agg_sanctions_summary` | Sanctions by registry & type | Roll-up |
| `analysis_compliance` | State ML-ready dataset | 27 states |
| `analysis_compliance_municipality` | Municipal ML-ready dataset | ~5,570 municipalities |
| `consolidated_clustering` | Normalized features for K-means | ~5,570 municipalities |

### 4.2 Feature Engineering

Features are normalized to `[0, 1]` via min-max scaling (Géron, 2022). Percentage and percentage-point deltas (2010 → 2022) capture **municipal trajectories**, not just snapshots.


In [ ]:
gold_datasets = {}
gold_sources = [
    ('agg_municipality_socioeconomic',   'Municipal features'),
    ('agg_state_summary',                'State aggregates'),
    ('agg_sanctions_summary',            'Sanctions rollup'),
    ('analysis_compliance',              'State ML dataset'),
    ('analysis_compliance_municipality', 'Municipal ML dataset'),
    ('consolidated_clustering',          'Clustering features'),
]
print("GOLD LAYER - Loading analysis-ready datasets")
print("=" * 60)
for name, desc in gold_sources:
    p = gold_dir / name / 'data.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        gold_datasets[name] = df
        print(f"  [OK] {name:40s} {len(df):>6,} rows x {len(df.columns):>3} cols - {desc}")
    else:
        print(f"  [!!] {name:40s} NOT FOUND")

df_city        = gold_datasets['analysis_compliance_municipality'].copy()
df_state       = gold_datasets['analysis_compliance'].copy()
df_cluster_raw = gold_datasets['consolidated_clustering'].copy()

print(f"\nMunicipal: {df_city.shape}  State: {df_state.shape}  Cluster: {df_cluster_raw.shape}")


---

## 5. Exploratory Data Analysis (EDA)

### 5.1 Methodological Justification

Tukey's **Exploratory Data Analysis** tradition (Tukey, 1977) motivates descriptive visualisations before any inferential modelling: distribution shapes, outliers, missingness patterns, bivariate associations (Field, 2017; Hair et al., 2018; McKinney, 2022).


In [ ]:
print("STATE-LEVEL DESCRIPTIVES")
print("=" * 60)
state_cols = [c for c in ['total_transfers', 'n_sanctions', 'avg_income_2022',
                          'literacy_rate_2022', 'sanctions_per_100k']
              if c in df_state.columns]
if state_cols:
    print(df_state[state_cols].describe().round(2))

print("\nMUNICIPAL-LEVEL DESCRIPTIVES")
print("=" * 60)
muni_cols = [c for c in ['total_transfers', 'n_sanctions',
                         'avg_income_real_2022_2022_brl', 'literacy_rate_2022',
                         'sanctions_per_million_brl_transfers']
             if c in df_city.columns]
if muni_cols:
    print(df_city[muni_cols].describe().round(2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if 'avg_income_real_2022_2022_brl' in df_city.columns:
    axes[0,0].hist(df_city['avg_income_real_2022_2022_brl'].dropna(),
                   bins=60, color='steelblue', edgecolor='white')
    axes[0,0].set_title('Average Real Income (2022 BRL)')
    axes[0,0].set_xlabel('BRL')

if 'literacy_rate_2022' in df_city.columns:
    axes[0,1].hist(df_city['literacy_rate_2022'].dropna(),
                   bins=60, color='darkorange', edgecolor='white')
    axes[0,1].set_title('Literacy Rate 2022 (%)')
    axes[0,1].set_xlabel('%')

if 'total_transfers' in df_city.columns:
    v = df_city['total_transfers'].dropna(); v = v[v > 0]
    axes[1,0].hist(np.log10(v), bins=60, color='seagreen', edgecolor='white')
    axes[1,0].set_title('Total Federal Transfers (log10 BRL)')
    axes[1,0].set_xlabel('log10(BRL)')

if 'sanctions_per_million_brl_transfers' in df_city.columns:
    v = df_city['sanctions_per_million_brl_transfers'].dropna()
    v = v[(v >= 0) & (v <= v.quantile(0.99))]
    axes[1,1].hist(v, bins=60, color='indianred', edgecolor='white')
    axes[1,1].set_title('Sanctions per Million BRL (99th pct cap)')
    axes[1,1].set_xlabel('Sanctions / M-BRL')

plt.suptitle('Municipal Distributions', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(output_dir / 'eda_distributions.png', dpi=120)
plt.show()


---

## 6. Statistical Analysis — OLS with Regional Controls

### 6.1 Methodological Justification

**Ordinary Least Squares (OLS)** regression is the workhorse of applied econometrics (Montgomery, Peck & Vining, 2012; Fox, 2015). When cross-sectional data exhibit regional heterogeneity, **regional fixed effects** via dummy variables are standard (Wooldridge, 2020). In Brazil, Norte/Nordeste/Sudeste/Sul/Centro-Oeste dummies are routinely used (Ferraz & Finan, 2008).

Assumption checks:
- **Normality of residuals** — Jarque-Bera
- **Heteroscedasticity** — Breusch-Pagan (Breusch & Pagan, 1979)
- **Multicollinearity** — VIF, threshold > 10 (Montgomery et al., 2012)

### 6.2 Correlation Analysis

Pearson (linear) + Spearman (rank-monotonic, robust to outliers) (Cohen et al., 2003).


In [ ]:
print("STATE-LEVEL CORRELATIONS (Pearson)")
print("=" * 60)
corr_vars = [c for c in ['total_transfers', 'n_sanctions', 'avg_income_2022',
                         'literacy_rate_2022', 'sanctions_per_100k']
             if c in df_state.columns]
if corr_vars:
    corr_mat = df_state[corr_vars].corr(method='pearson')
    print(corr_mat.round(3))

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title('State-Level Correlations')
    plt.tight_layout()
    plt.savefig(output_dir / 'correlations_state.png', dpi=120)
    plt.show()


In [ ]:
if 'sanctions_per_100k' in df_state.columns and 'avg_income_2022' in df_state.columns:
    df_reg = df_state.dropna(subset=['sanctions_per_100k', 'avg_income_2022']).copy()
    if 'region' in df_reg.columns:
        region_dummies = pd.get_dummies(df_reg['region'], prefix='region', drop_first=True)
        X = pd.concat([df_reg[['avg_income_2022']], region_dummies], axis=1)
    else:
        X = df_reg[['avg_income_2022']].copy()

    X = sm.add_constant(X).astype(float)
    y = df_reg['sanctions_per_100k'].astype(float)
    model = sm.OLS(y, X).fit()
    print(model.summary())

    print("\nDIAGNOSTICS")
    print("=" * 60)
    from scipy.stats import jarque_bera
    jb_s, jb_p = jarque_bera(model.resid)
    print(f"Jarque-Bera:   stat={jb_s:.3f}, p={jb_p:.4f}  ({'Normal' if jb_p > 0.05 else 'Non-normal'})")

    bp_s, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
    print(f"Breusch-Pagan: stat={bp_s:.3f}, p={bp_p:.4f}  ({'Homosc.' if bp_p > 0.05 else 'Heterosc.'})")

    print("\nVIF:")
    for i, col in enumerate(X.columns):
        if col == 'const': continue
        try:
            v = variance_inflation_factor(X.values, i)
            flag = '  >10 WARNING' if v > 10 else ''
            print(f"  {col:30s} VIF = {v:6.2f}{flag}")
        except Exception: pass
else:
    print("Skipping OLS - required columns missing.")


---

## 7. Machine Learning — Predictive Modeling

### 7.1 Methodological Justification

We complement OLS with:
- **ElasticNet** (Zou & Hastie, 2005): L1+L2 penalty, robust to multicollinearity, implicit feature selection.
- **Random Forest** (Breiman, 2001): ensemble of trees, captures non-linearities, permutation feature importance (Molnar, 2022).

Validation: hold-out 80/20 split, metrics R², MAE, RMSE (Kuhn & Johnson, 2013; Hastie et al., 2009).


In [ ]:
ml_features = [c for c in ['avg_income_real_2022_2022_brl', 'literacy_rate_2022',
                           'population_2022', 'households_2022', 'total_transfers']
               if c in df_city.columns]
target = 'n_sanctions' if 'n_sanctions' in df_city.columns else None

if ml_features and target and target in df_city.columns:
    df_ml = df_city[ml_features + [target]].dropna()
    print(f"ML dataset: {len(df_ml):,} rows x {len(ml_features)} features")

    X = df_ml[ml_features].values
    y = df_ml[target].values
    X_scaled = StandardScaler().fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=SEED)

    en = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=SEED, max_iter=5000).fit(X_tr, y_tr)
    rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)

    def metrics(y_true, y_pred):
        return (r2_score(y_true, y_pred),
                mean_absolute_error(y_true, y_pred),
                float(np.sqrt(mean_squared_error(y_true, y_pred))))

    en_r2, en_mae, en_rmse = metrics(y_te, en.predict(X_te))
    rf_r2, rf_mae, rf_rmse = metrics(y_te, rf.predict(X_te))

    print(f"\n{'Model':<20} {'R2':>8} {'MAE':>10} {'RMSE':>10}")
    print("-" * 60)
    print(f"{'ElasticNet':<20} {en_r2:>8.3f} {en_mae:>10.3f} {en_rmse:>10.3f}")
    print(f"{'Random Forest':<20} {rf_r2:>8.3f} {rf_mae:>10.3f} {rf_rmse:>10.3f}")

    print("\nRandom Forest feature importance:")
    imp = pd.DataFrame({'feature': ml_features, 'importance': rf.feature_importances_})
    for _, r in imp.sort_values('importance', ascending=False).iterrows():
        print(f"  {r['feature']:35s} {r['importance']:.4f}")
else:
    print("Skipping ML - insufficient features/target.")


---

## 8. Clustering Analysis — K-means (K=4) with PCA

### 8.1 Methodological Justification

**K-means** (MacQueen, 1967) partitions municipalities into homogeneous groups. K = 4 is fixed based on:
- Elbow + silhouette (Rousseeuw, 1987; James et al., 2021)
- Substantive interpretability (four municipal archetypes)

**PCA** (Jolliffe, 2002) reduces the 8-dim feature space to 3 components for visualization.

### 8.2 Features Used (8, min-max normalized)

1. `population_2022_norm`
2. `literacy_rate_2022_norm`
3. `avg_income_real_2022_2022_brl_norm`
4. `households_2022_norm`
5. `population_change_pct_norm`
6. `literacy_change_pp_norm`
7. `income_change_real_pct_norm`
8. `households_change_pct_norm`

Combines **levels** (2022) + **trajectories** (2010→2022 deltas).


In [ ]:
cluster_features = [
    'population_2022_norm', 'literacy_rate_2022_norm',
    'avg_income_real_2022_2022_brl_norm', 'households_2022_norm',
    'population_change_pct_norm', 'literacy_change_pp_norm',
    'income_change_real_pct_norm', 'households_change_pct_norm',
]
cluster_features = [f for f in cluster_features if f in df_cluster_raw.columns]
print(f"Using {len(cluster_features)} clustering features")

df_cluster = df_cluster_raw.copy()
X_c = df_cluster[cluster_features].dropna()
valid_idx = X_c.index

if 'cluster' not in df_cluster.columns or df_cluster['cluster'].isna().all():
    print("Computing K-means (K=4)...")
    kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10)
    df_cluster.loc[valid_idx, 'cluster'] = kmeans.fit_predict(X_c)
    from sklearn.metrics import silhouette_score
    print(f"Silhouette: {silhouette_score(X_c, df_cluster.loc[valid_idx, 'cluster']):.4f}")

if 'PC1' not in df_cluster.columns or df_cluster['PC1'].isna().all():
    pca = PCA(n_components=3)
    pc = pca.fit_transform(X_c)
    df_cluster.loc[valid_idx, 'PC1'] = pc[:, 0]
    df_cluster.loc[valid_idx, 'PC2'] = pc[:, 1]
    df_cluster.loc[valid_idx, 'PC3'] = pc[:, 2]
    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

print("\nCluster sizes:")
for c, n in df_cluster['cluster'].value_counts().sort_index().items():
    print(f"  Cluster {int(c)}: {n:>5,} ({100*n/len(df_cluster):5.1f}%)")


In [ ]:
print("CLUSTER PROFILES (means of raw features)")
print("=" * 60)
prof_feats = [c for c in ['population_2022', 'literacy_rate_2022',
                          'avg_income_real_2022_2022_brl', 'households_2022']
              if c in df_cluster.columns]
if prof_feats:
    print(df_cluster.groupby('cluster')[prof_feats].mean().round(2))

fig, ax = plt.subplots(figsize=(10, 7))
for c in sorted(df_cluster['cluster'].dropna().unique()):
    sub = df_cluster[df_cluster['cluster'] == c]
    ax.scatter(sub['PC1'], sub['PC2'], label=f'Cluster {int(c)}', s=10, alpha=0.55)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('K-means Clusters in PCA Space (K=4)')
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / 'clusters_pca.png', dpi=120)
plt.show()


---

## 9. Corruption vs HDI — Cluster-Stratified Correlation

### 9.1 Research Motivation

The central research question asks whether corruption contributes to lower HDI. We operationalize:

- **Corruption proxy (X)**: `sanctions_per_million_brl_transfers` — count of administrative sanctions (CEIS, CNEP, CEPIM) normalized by federal BRL transferred. Sanction density signals inefficiency (Ferraz & Finan, 2008; Olken & Pande, 2012). **Caveat**: sanctions reflect both malfeasance *and* **detection capacity** — itself correlated with state capacity (Ferraz & Finan, 2008).

- **HDI components (Y)**: `avg_income_real_2022_2022_brl`, `literacy_rate_2022`.

### 9.2 Why Cluster-Stratified?

A naive bivariate correlation across ~5,570 municipalities confounds the **corruption → HDI** effect with the **development-detection** confounder. By **conditioning on cluster** (stratification on similar socioeconomic profiles), we approximate an "apples-to-apples" comparison, following the matching tradition (Rosenbaum & Rubin, 1983; Stuart, 2010). This is **not** a causal identification strategy — it is a descriptive stratified correlation that reduces (but does not eliminate) confounding.

### 9.3 Vulnerability Index

$$ V = \log(1 + \text{sanctions\_per\_M\_BRL}) - 5 \cdot \text{HDI}_{\text{proxy}} $$

where HDI_proxy ∈ [0, 1] is the average of normalized income and literacy. Higher V = higher sanctions + lower development.


In [ ]:
df_analysis = df_city.merge(
    df_cluster[['municipality_code', 'cluster', 'PC1', 'PC2']],
    on='municipality_code', how='left'
)

CORRUPTION_VAR = 'sanctions_per_million_brl_transfers'
HDI_VARS = [c for c in ['avg_income_real_2022_2022_brl', 'literacy_rate_2022', 'avg_income_2022']
            if c in df_analysis.columns]

df_analysis['has_sanctions_data'] = (
    df_analysis[CORRUPTION_VAR].notna()
    & (df_analysis[CORRUPTION_VAR] >= 0)
    & df_analysis['total_transfers'].notna()
    & (df_analysis['total_transfers'] > 0)
)

n_total = len(df_analysis)
n_valid = int(df_analysis['has_sanctions_data'].sum())
print(f"Total municipalities:      {n_total:,}")
print(f"With valid sanctions data: {n_valid:,} ({100*n_valid/n_total:.1f}%)")
print(f"Clusters:                  {df_analysis['cluster'].nunique()}")


In [ ]:
def cluster_correlation(df, cid, x, y):
    sub = df[(df['cluster'] == cid) & df['has_sanctions_data']
             & df[x].notna() & df[y].notna()]
    if len(sub) < 10:
        return {'n': len(sub), 'pearson_r': None, 'pearson_p': None,
                'spearman_r': None, 'spearman_p': None, 'significance': 'n<10'}
    z = np.abs(stats.zscore(sub[x]))
    clean = sub[z < 3]
    if len(clean) < 10:
        return {'n': len(clean), 'pearson_r': None, 'pearson_p': None,
                'spearman_r': None, 'spearman_p': None, 'significance': 'n<10 clean'}
    rp, pp = pearsonr(clean[x], clean[y])
    rs, ps = spearmanr(clean[x], clean[y])
    sig = '***' if pp < 0.001 else '**' if pp < 0.01 else '*' if pp < 0.05 else 'ns'
    return {'n': int(len(clean)), 'pearson_r': round(rp, 4), 'pearson_p': round(pp, 6),
            'spearman_r': round(rs, 4), 'spearman_p': round(ps, 6), 'significance': sig}

results = []
for cid in sorted(df_analysis['cluster'].dropna().unique()):
    for hdi in HDI_VARS:
        r = cluster_correlation(df_analysis, cid, CORRUPTION_VAR, hdi)
        r['cluster'] = int(cid); r['hdi_variable'] = hdi
        results.append(r)

df_correlations = pd.DataFrame(results)
print("CLUSTER-STRATIFIED CORRELATIONS")
print("=" * 70)
cols = ['cluster', 'hdi_variable', 'n', 'pearson_r', 'pearson_p', 'spearman_r', 'significance']
print(df_correlations[cols].to_string(index=False))
sig = df_correlations[df_correlations['pearson_p'].notna() & (df_correlations['pearson_p'] < 0.05)]
print(f"\nSignificant (p<0.05): {len(sig)} of {len(df_correlations)} pairs")


In [ ]:
def vulnerability_index(row):
    if pd.isna(row[CORRUPTION_VAR]) or row[CORRUPTION_VAR] < 0: return None
    log_s = np.log1p(row[CORRUPTION_VAR])
    inc = row.get('avg_income_real_2022_2022_brl', 0) or 0
    lit = row.get('literacy_rate_2022', 0) or 0
    hdi = min(max(((inc / 5000.0) + (lit / 100.0)) / 2.0, 0.0), 1.0)
    return log_s - (hdi * 5.0)

df_analysis['vulnerability_index'] = df_analysis.apply(vulnerability_index, axis=1)
print("VULNERABILITY INDEX — summary by cluster")
print("=" * 60)
print(df_analysis.groupby('cluster')['vulnerability_index'].agg(
    ['count', 'mean', 'std', 'min', 'max']).round(3))


In [ ]:
TOP_N = 5
reps = []
for cid in sorted(df_analysis['cluster'].dropna().unique()):
    cd = df_analysis[(df_analysis['cluster'] == cid) & df_analysis['vulnerability_index'].notna()]
    best = cd.nsmallest(TOP_N, 'vulnerability_index').copy()
    best['category'] = 'Best Management'
    best['rank_in_cluster'] = range(1, len(best) + 1)
    worst = cd.nlargest(TOP_N, 'vulnerability_index').copy()
    worst['category'] = 'High Vulnerability'
    worst['rank_in_cluster'] = range(1, len(worst) + 1)
    reps.extend([best, worst])

df_sample = pd.concat(reps, ignore_index=True)
print(f"Representative sample: {len(df_sample)} cities")
preview = [c for c in ['municipality_name', 'state_name', 'cluster', 'category',
                       'vulnerability_index', 'sanctions_per_million_brl_transfers',
                       'avg_income_real_2022_2022_brl', 'literacy_rate_2022']
           if c in df_sample.columns]
print(df_sample[preview].head(20).to_string(index=False))


---

## 10. QGIS Map Generation — GeoJSON Export

### 10.1 Methodological Justification

Geospatial visualization is essential for heterogeneous municipal patterns (Bivand, Pebesma & Gómez-Rubio, 2013). We export **GeoJSON** (OGC standard) with municipality centroids tagged with the Vulnerability Index, cluster, and raw indicators. Consumable by **QGIS** (QGIS Development Team, 2024), Power BI, Tableau, or web GIS libraries.


In [ ]:
try:
    import shapefile  # pyshp
    HAS_SHAPEFILE = True
except ImportError:
    HAS_SHAPEFILE = False
    print("[!] pyshp not installed - skipping GeoJSON.")

if HAS_SHAPEFILE:
    from src.analysis.presentation_assets import download_ibge_boundary_zip
    zip_path = qgis_dir / 'BR_Municipios_2022.zip'
    if not zip_path.exists():
        print(f"Downloading BR_Municipios_2022.zip ...")
        zip_path = download_ibge_boundary_zip("BR_Municipios_2022.zip", output_dir=qgis_dir)
    tmp = tempfile.mkdtemp(prefix='ibge_muni_')
    try:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        shp_path = next(Path(tmp).glob('*.shp'))
        reader = shapefile.Reader(str(shp_path))
        fields = [f[0] for f in reader.fields[1:]]
        code_idx = fields.index('CD_MUN')
        cents = []
        for rec in reader.iterShapeRecords():
            code = str(rec.record[code_idx]).zfill(7)
            xmin, ymin, xmax, ymax = rec.shape.bbox
            cents.append({'municipality_code': code,
                          'lon': (xmin+xmax)/2.0, 'lat': (ymin+ymax)/2.0})
        reader.close()
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    df_geo = df_analysis.merge(pd.DataFrame(cents), on='municipality_code', how='inner')

    def categorize(v):
        if pd.isna(v): return 'No data'
        if v < -3: return 'Very Low (Blue)'
        if v < -1: return 'Low (Light Blue)'
        if v < 1:  return 'Neutral (Yellow)'
        if v < 3:  return 'High (Orange)'
        return 'Very High (Red)'

    df_geo['vulnerability_category'] = df_geo['vulnerability_index'].apply(categorize)

    features = []
    for _, r in df_geo.iterrows():
        if pd.isna(r['lon']) or pd.isna(r['lat']): continue
        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(r['lon']), float(r['lat'])]},
            'properties': {
                'municipality_code': str(r['municipality_code']),
                'municipality_name': str(r.get('municipality_name', '')),
                'state_code': str(r['state_code']) if pd.notna(r.get('state_code')) else None,
                'state_name': str(r['state_name']) if pd.notna(r.get('state_name')) else None,
                'cluster': int(r['cluster']) if pd.notna(r.get('cluster')) else None,
                'vulnerability_index': float(r['vulnerability_index']) if pd.notna(r.get('vulnerability_index')) else None,
                'vulnerability_category': str(r['vulnerability_category']),
                'sanctions_per_million': float(r[CORRUPTION_VAR]) if pd.notna(r[CORRUPTION_VAR]) else 0.0,
                'avg_income_2022': float(r['avg_income_real_2022_2022_brl']) if pd.notna(r.get('avg_income_real_2022_2022_brl')) else None,
                'literacy_rate_2022': float(r['literacy_rate_2022']) if pd.notna(r.get('literacy_rate_2022')) else None,
                'total_transfers': float(r['total_transfers']) if pd.notna(r.get('total_transfers')) else 0.0,
                'n_sanctions': int(r['n_sanctions']) if pd.notna(r.get('n_sanctions')) else 0,
            }
        })

    geojson = {'type': 'FeatureCollection',
               'name': 'Brazil_Municipalities_Vulnerability_Index',
               'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
               'features': features}
    out = qgis_dir / 'brazil_municipalities_vulnerability_index.geojson'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)
    print(f"[OK] GeoJSON: {out}")
    print(f"     Features: {len(features):,}")
    print(f"     Size: {out.stat().st_size/(1024**2):.2f} MB")

    # ── In-notebook interactive vulnerability map ─────────────────────────────
    import plotly.express as px

    _map_df = df_geo.dropna(subset=["vulnerability_index", "lat", "lon"]).copy()
    _map_df["cluster_label"] = _map_df["cluster"].apply(
        lambda x: f"Cluster {int(x)}" if pd.notna(x) else "Unknown"
    )
    fig_vuln_map = px.scatter_mapbox(
        _map_df,
        lat="lat", lon="lon",
        color="vulnerability_index",
        color_continuous_scale="RdYlBu_r",
        hover_name="municipality_name",
        hover_data={
            "state_name": True,
            "cluster_label": True,
            "vulnerability_index": ":.3f",
            "sanctions_per_million_brl_transfers": ":.2f",
            "avg_income_real_2022_2022_brl": ":,.0f",
            "literacy_rate_2022": ":.1f",
            "lat": False, "lon": False,
        },
        mapbox_style="carto-positron",
        zoom=3.2,
        center={"lat": -14.2, "lon": -52.9},
        title="Municipal Vulnerability Index — Sanctions/Transfer vs HDI Proxy (Red = High Risk)",
        labels={"vulnerability_index": "Vulnerability"},
    )
    fig_vuln_map.update_traces(marker=dict(size=5, opacity=0.75))
    fig_vuln_map.update_layout(
        height=680,
        margin={"r": 0, "t": 60, "l": 0, "b": 0},
        coloraxis_colorbar=dict(title="Vulnerability"),
    )
    fig_vuln_map.show()
    print(f"Map rendered: {len(_map_df):,} municipalities plotted")


---

## 11. Dashboard Export — Power BI / Tableau

### 11.1 Methodological Justification

BI tools consume **tidy CSVs** (Wickham, 2014). We export the minimal set needed for interactive dashboards at the defense (Davenport & Harris, 2017; Provost & Fawcett, 2013).


In [ ]:
df_correlations.to_csv(output_dir / 'correlation_by_cluster.csv', index=False)

sample_cols = [c for c in ['municipality_code', 'municipality_name', 'state_code', 'state_name',
                           'cluster', 'category', 'rank_in_cluster', 'vulnerability_index',
                           'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl',
                           'literacy_rate_2022', 'total_transfers', 'n_sanctions']
               if c in df_sample.columns]
df_sample[sample_cols].to_csv(output_dir / 'representative_sample_cities.csv', index=False)
df_sample[sample_cols].to_csv(output_dir / 'amostra_representativa_cidades.csv', index=False)

numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
summary_cols = [c for c in ['vulnerability_index', CORRUPTION_VAR,
                            'avg_income_real_2022_2022_brl', 'literacy_rate_2022']
                if c in numeric_cols]
cluster_summary = df_analysis.groupby('cluster')[summary_cols].agg(
    ['count', 'mean', 'std', 'min', 'max']).round(3)
cluster_summary.to_csv(output_dir / 'cluster_summary.csv')
cluster_summary.to_csv(output_dir / 'resumo_por_cluster.csv')

with open(output_dir / 'representative_sample.json', 'w', encoding='utf-8') as f:
    json.dump(df_sample.to_dict('records'), f, ensure_ascii=False, indent=2, default=str)

print("Dashboard exports:")
for fname in ['correlation_by_cluster.csv', 'representative_sample_cities.csv',
              'amostra_representativa_cidades.csv', 'cluster_summary.csv',
              'resumo_por_cluster.csv', 'representative_sample.json']:
    fp = output_dir / fname
    if fp.exists():
        print(f"  [OK] {fname:<42s} ({fp.stat().st_size:,} bytes)")

# ── Interactive Dashboard ─────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_dash = df_analysis[df_analysis["vulnerability_index"].notna()].copy()
_dash["cluster_label"] = _dash["cluster"].apply(
    lambda x: f"Cluster {int(x)}" if pd.notna(x) else "Unknown"
)
_clusters = sorted(_dash["cluster"].dropna().unique())
_palette = px.colors.qualitative.Set2

fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Vulnerability Index by Cluster",
        "Sanctions/Million BRL vs Income by Cluster",
        "Pearson r Heatmap (Cluster × HDI Variable)",
        "Top 20 Most Vulnerable Municipalities",
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.12,
)

# 1. Box: vulnerability by cluster
for i, c in enumerate(_clusters):
    fig_dash.add_trace(
        go.Box(
            y=_dash[_dash["cluster"] == c]["vulnerability_index"],
            name=f"Cluster {int(c)}",
            marker_color=_palette[i % len(_palette)],
            showlegend=False,
        ),
        row=1, col=1,
    )
fig_dash.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

# 2. Scatter: sanctions vs income
for i, c in enumerate(_clusters):
    _s = _dash[_dash["cluster"] == c]
    _inc_col = "avg_income_real_2022_2022_brl"
    if _inc_col in _s.columns:
        fig_dash.add_trace(
            go.Scatter(
                x=_s[_inc_col],
                y=_s[CORRUPTION_VAR],
                mode="markers",
                marker=dict(size=4, opacity=0.5, color=_palette[i % len(_palette)]),
                name=f"Cluster {int(c)}",
                text=_s.get("municipality_name", _s.index.astype(str)),
                hovertemplate="%{text}<br>Income: %{x:,.0f}<br>Sanctions/M: %{y:.2f}<extra></extra>",
            ),
            row=1, col=2,
        )

# 3. Correlation heatmap
if len(df_correlations) > 0:
    _idh_label = {
        "avg_income_real_2022_2022_brl": "Income",
        "literacy_rate_2022": "Literacy",
        "avg_income_2022": "Income (raw)",
    }
    _piv = df_correlations.pivot_table(index="cluster", columns="hdi_variable", values="pearson_r")
    _piv.columns = [_idh_label.get(c, c) for c in _piv.columns]
    fig_dash.add_trace(
        go.Heatmap(
            z=_piv.values,
            x=list(_piv.columns),
            y=[f"Cluster {int(c)}" for c in _piv.index],
            colorscale="RdBu", zmid=0, zmin=-0.5, zmax=0.5,
            text=_piv.values.round(3), texttemplate="%{text}",
            showscale=False,
        ),
        row=2, col=1,
    )

# 4. Top-20 most vulnerable municipalities
_name_col = next((c for c in ["municipality_name"] if c in _dash.columns), None)
_state_col = next((c for c in ["state_name"] if c in _dash.columns), None)
_top20 = _dash.nlargest(20, "vulnerability_index").copy()
_label = (
    (_top20[_name_col] + " (" + _top20[_state_col] + ")")
    if _name_col and _state_col
    else _top20.index.astype(str)
)
fig_dash.add_trace(
    go.Bar(
        x=_top20["vulnerability_index"],
        y=_label,
        orientation="h",
        marker_color="crimson",
        text=_top20["cluster_label"],
        textposition="outside",
        showlegend=False,
    ),
    row=2, col=2,
)

fig_dash.update_layout(
    height=900,
    title_text="Consolidated Dashboard — Vulnerability Analysis by Cluster",
    title_font_size=16,
    showlegend=True,
    legend=dict(title="Cluster", x=1.01),
)
fig_dash.update_xaxes(title_text="Average Income (BRL)", row=1, col=2)
fig_dash.update_yaxes(title_text="Sanctions / Million BRL", row=1, col=2)
fig_dash.update_xaxes(title_text="Vulnerability Index", row=2, col=2)
fig_dash.show()


---

## 12. Conclusions

### 12.1 Main Findings

1. **State-level**: Strong Pearson correlation (r ≈ 0.74, p < 0.001) between average municipal income (2022) and sanctions per 100k inhabitants — consistent with **detection-bias hypothesis** (Ferraz & Finan, 2008).

2. **OLS with regional dummies**: R² ≈ 0.835 for state-level sanctions per 100k. Norte/Nordeste effect: β ≈ +22 after controlling for income → structural regional factors beyond income.

3. **Municipal-level**: Much weaker (r ≈ 0.15 at n ≈ 5,570). Heterogeneity dominates.

4. **K-means (K=4)**: Silhouette ≈ 0.29, PCA-3 variance ≈ 84%. Four archetypes: small-rural, medium-interior, medium-capital, metropolitan.

5. **Cluster-stratified correlations**: Small and mostly non-significant within clusters. When controlling for socioeconomic similarity, the residual sanctions-per-BRL signal carries little information about HDI. Consistent with the view (Olken & Pande, 2012) that sanction counts are a **noisy** and **biased** corruption measure.

6. **Vulnerability Index**: Identifies municipalities with high sanction density + low HDI — **red flags** for qualitative investigation (TCU, 2023).

### 12.2 Limitations

- **Endogenous sanctions**: higher-capacity states audit more (Ferraz & Finan, 2008).
- **Cross-sectional design**: no pre/post panel (Angrist & Pischke, 2009).
- **HDI proxy**: income + literacy, not full PNUD-IDH-M.
- **Aggregation unit**: masks intra-municipal heterogeneity.

### 12.3 Policy Implications

- **Data strategy**: Silver-layer enrichment is a **pre-condition** for meaningful analysis; investment in interoperable open data is warranted (Brasil, 2011; CGU, 2024).
- **Audit targeting**: Vulnerability Index can prioritize audit targets — aligned with TCU's risk-based framework (TCU, 2023).
- **Heterogeneity matters**: anti-corruption policies should be **stratified by cluster**, not uniform.

### 12.4 Future Work

- Quasi-experimental ID: DiD with exogenous variation in transfer rules (Angrist & Pischke, 2009).
- Panel data: multi-year municipal IDH-M.
- Spatial econometrics: explicit spatial autocorrelation (Anselin, 1988).
- Alternative corruption proxies: CGU audit findings, contract anomaly detection.


---

## 13. References (ABNT-compatible)

### 13.1 Data Architecture & Engineering

ARMBRUST, M. et al. Delta Lake: High-Performance ACID Table Storage over Cloud Object Stores. **Proceedings of the VLDB Endowment**, v. 13, n. 12, p. 3411-3424, 2020. DOI: 10.14778/3415478.3415560.

DATABRICKS. **What is a Medallion Architecture?** Databricks Documentation, 2023.

INMON, W. H. **Building the Data Warehouse**. 4. ed. Hoboken, NJ: Wiley, 2005.

KIMBALL, R.; ROSS, M. **The Data Warehouse Toolkit**. 3. ed. Hoboken, NJ: Wiley, 2013.

KLEPPMANN, M. **Designing Data-Intensive Applications**. Sebastopol: O'Reilly, 2017.

### 13.2 Statistical Methods & Econometrics

ANGRIST, J. D.; PISCHKE, J.-S. **Mostly Harmless Econometrics**. Princeton: Princeton University Press, 2009.

BREUSCH, T. S.; PAGAN, A. R. A Simple Test for Heteroscedasticity. **Econometrica**, v. 47, n. 5, p. 1287-1294, 1979.

COHEN, J.; COHEN, P.; WEST, S. G.; AIKEN, L. S. **Applied Multiple Regression/Correlation Analysis**. 3. ed. Routledge, 2003.

FIELD, A. **Discovering Statistics Using IBM SPSS Statistics**. 5. ed. London: SAGE, 2017.

FOX, J. **Applied Regression Analysis and Generalized Linear Models**. 3. ed. SAGE, 2015.

MONTGOMERY, D. C.; PECK, E. A.; VINING, G. G. **Introduction to Linear Regression Analysis**. 5. ed. Wiley, 2012.

ROSENBAUM, P. R.; RUBIN, D. B. The central role of the propensity score in observational studies. **Biometrika**, v. 70, n. 1, p. 41-55, 1983.

STUART, E. A. Matching Methods for Causal Inference: A Review and a Look Forward. **Statistical Science**, v. 25, n. 1, p. 1-21, 2010.

TUKEY, J. W. **Exploratory Data Analysis**. Reading, MA: Addison-Wesley, 1977.

WOOLDRIDGE, J. M. **Econometric Analysis of Cross Section and Panel Data**. 2. ed. MIT Press, 2020.

### 13.3 Machine Learning & Multivariate Analysis

BREIMAN, L. Random Forests. **Machine Learning**, v. 45, n. 1, p. 5-32, 2001.

GÉRON, A. **Hands-On Machine Learning**. 3. ed. O'Reilly, 2022.

HAIR, J. F. et al. **Multivariate Data Analysis**. 8. ed. Cengage, 2018.

HASTIE, T.; TIBSHIRANI, R.; FRIEDMAN, J. **The Elements of Statistical Learning**. 2. ed. Springer, 2009.

JAMES, G.; WITTEN, D.; HASTIE, T.; TIBSHIRANI, R. **An Introduction to Statistical Learning**. 2. ed. Springer, 2021.

JOLLIFFE, I. T. **Principal Component Analysis**. 2. ed. Springer, 2002.

KUHN, M.; JOHNSON, K. **Applied Predictive Modeling**. Springer, 2013.

MACQUEEN, J. B. Some Methods for Classification. **Berkeley Symposium**, v. 1, p. 281-297, 1967.

MOLNAR, C. **Interpretable Machine Learning**. 2. ed. 2022.

PEDREGOSA, F. et al. Scikit-learn: Machine Learning in Python. **JMLR**, v. 12, p. 2825-2830, 2011.

ROUSSEEUW, P. J. Silhouettes. **J. Comp. Appl. Math.**, v. 20, p. 53-65, 1987.

ZOU, H.; HASTIE, T. Regularization and variable selection via the Elastic Net. **JRSS-B**, v. 67, n. 2, p. 301-320, 2005.

### 13.4 Public Administration & Corruption

AFONSO, A.; SCHUKNECHT, L.; TANZI, V. Public Sector Efficiency. **Public Choice**, v. 123, n. 3-4, p. 321-347, 2005.

FERRAZ, C.; FINAN, F. Exposing Corrupt Politicians. **QJE**, v. 123, n. 2, p. 703-745, 2008.

OLKEN, B. A.; PANDE, R. Corruption in Developing Countries. **Annual Review of Economics**, v. 4, p. 479-509, 2012.

ROSE-ACKERMAN, S.; PALIFKA, B. J. **Corruption and Government**. 2. ed. Cambridge UP, 2016.

### 13.5 Brazilian Public Sector

BRASIL. **Lei nº 12.527/2011 — Lei de Acesso à Informação**. Brasília: Presidência da República, 2011.

BRASIL. **Lei nº 13.709/2018 — LGPD**. Brasília: Presidência da República, 2018.

CGU. **Portal da Transparência do Governo Federal**. Brasília: CGU, 2024.

IBGE. **Censo Demográfico 2010**. Rio de Janeiro: IBGE, 2010.

IBGE. **Censo Demográfico 2022**. Rio de Janeiro: IBGE, 2022.

IBGE. **API SIDRA**. Rio de Janeiro: IBGE, 2024.

STN. **SICONFI**. Brasília: STN, 2024.

TCU. **Referencial de Combate à Fraude e Corrupção**. Brasília: TCU, 2023.

PNUD. **Human Development Report 2020**. New York: UNDP, 2020.

### 13.6 Geospatial & Reproducibility

ANSELIN, L. **Spatial Econometrics**. Springer, 1988.

BIVAND, R. S.; PEBESMA, E.; GÓMEZ-RUBIO, V. **Applied Spatial Data Analysis with R**. 2. ed. Springer, 2013.

GANDRUD, C. **Reproducible Research with R and RStudio**. 3. ed. CRC Press, 2020.

QGIS DEVELOPMENT TEAM. **QGIS Geographic Information System**. OSGeo, 2024.

WICKHAM, H. Tidy Data. **J. Stat. Software**, v. 59, n. 10, 2014.

WICKHAM, H.; GROLEMUND, G. **R for Data Science**. O'Reilly, 2017.

### 13.7 Python Scientific Stack

HARRIS, C. R. et al. Array programming with NumPy. **Nature**, v. 585, p. 357-362, 2020.

HUNTER, J. D. Matplotlib: A 2D Graphics Environment. **CiSE**, v. 9, n. 3, p. 90-95, 2007.

MCKINNEY, W. Data Structures for Statistical Computing in Python. **SciPy Proceedings**, p. 56-61, 2010.

MCKINNEY, W. **Python for Data Analysis**. 3. ed. O'Reilly, 2022.

SEABOLD, S.; PERKTOLD, J. statsmodels. **SciPy Proceedings**, p. 92-96, 2010.

VANDERPLAS, J. **Python Data Science Handbook**. O'Reilly, 2016.

WASKOM, M. L. seaborn: statistical data visualization. **JOSS**, v. 6, n. 60, 2021.

---

## End of Notebook

This notebook is the **complete empirical evidence** for the TCC. All data, code, and conclusions are self-contained and reproducible.
